# EASM scan: wikimedia.org

Triggers a scan through the discovery-scheduler pod, tails the pipeline pod logs, and queries the Postgres gold tables.
Requires `kubectl` pointed at the local kind cluster (namespace `easm`) and `pandas`.

> The deployments currently run with `OFFLINE_MODE=true` (synthetic CT records and probes). For real discovery/probing of wikimedia.org set `LIVE = True` in section 1.

In [1]:
import json, subprocess, time
from io import StringIO
import pandas as pd

NS = "easm"
TENANT = "tenant_wikimedia"
DOMAIN = "wikimedia.org"

def kubectl(*args, check=True):
    r = subprocess.run(["kubectl", "-n", NS, *args], capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(r.stderr or r.stdout)
    return r.stdout

def sql(query):
    out = kubectl("exec", "deploy/postgres", "--", "psql", "-U", "easm", "-d", "easm", "--csv", "-c", query)
    return pd.read_csv(StringIO(out))

def logs(app, since="10m", tail=200, grep=None):
    """Logs from all pods labelled app=<name>, prefixed with the pod name."""
    out = kubectl("logs", "-l", f"app={app}", "--prefix", f"--since={since}", f"--tail={tail}", "--max-log-requests=10", check=False)
    lines = [l for l in out.splitlines() if not grep or grep in l]
    print("\n".join(lines) or f"(no logs for app={app})")

print(kubectl("get", "pods"))

NAME                                      READY   STATUS    RESTARTS      AGE
asset-api-67659b8fd6-jsn8f                1/1     Running   0             97m
discovery-scheduler-6454f9dcf-dr4fx       1/1     Running   0             49m
postgres-79967df69f-bgktx                 1/1     Running   1 (90m ago)   91m
probe-worker-7bbf47445f-8bfrd             1/1     Running   0             49m
probe-worker-7bbf47445f-q2h6z             1/1     Running   0             49m
probe-worker-7bbf47445f-xr245             1/1     Running   0             49m
redis-79cfc78c7-mbwcv                     1/1     Running   0             84m
redpanda-8f59ffbfb-5kw9j                  1/1     Running   1 (91m ago)   91m
spark-stream-processor-66769ddbfd-w5hvj   1/1     Running   0             54m



## 1. (Optional) Enable live mode
Switches scheduler and probe workers to `OFFLINE_MODE=false` (queries crt.sh and probes wikimedia.org hosts over DNS/TLS/HTTP). Patches the running deployments only; `k8s/02-apps.yaml` is unchanged.

In [2]:
LIVE = False
if LIVE:
    for d in ("discovery-scheduler", "probe-worker"):
        kubectl("set", "env", f"deploy/{d}", "OFFLINE_MODE=false")
        kubectl("rollout", "status", f"deploy/{d}", "--timeout=180s")

## 2. Trigger the scan

In [3]:
payload = json.dumps({"tenant_id": TENANT, "root_domain": DOMAIN})
script = (
    "import urllib.request as u;"
    f"r=u.Request('http://localhost:8000/scans',data={payload!r}.encode(),headers={{'Content-Type':'application/json'}});"
    "print(u.urlopen(r,timeout=120).read().decode())"
)
# crt.sh is occasionally down (502); the scheduler returns 500 when its retries are exhausted.
for attempt in range(1, 6):
    try:
        print(kubectl("exec", "deploy/discovery-scheduler", "--", "python", "-c", script))
        break
    except RuntimeError as e:
        print(f"attempt {attempt} failed (crt.sh likely unavailable): {str(e).splitlines()[-2] if len(str(e).splitlines())>1 else e}")
        time.sleep(30)
else:
    print("Scan not triggered; continuing with assets from earlier scans (if any).")

RuntimeError: Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/urllib/request.py", line 215, in urlopen
    return opener.open(url, data, timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/urllib/request.py", line 521, in open
    response = meth(req, response)
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/urllib/request.py", line 630, in http_response
    response = self.parent.error(
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/urllib/request.py", line 559, in error
    return self._call_chain(*args)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/urllib/request.py", line 492, in _call_chain
    result = func(*args)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/urllib/request.py", line 639, in http_error_default
    raise HTTPError(req.full_url, code, msg, hdrs, fp)
urllib.error.HTTPError: HTTP Error 500: Interna

## 3. Pod logs

In [ ]:
print("=== discovery-scheduler ==="); logs("discovery-scheduler", since="5m")
print("\n=== probe-worker (x3) ==="); logs("probe-worker", since="5m")

In [ ]:
print("=== spark-stream-processor ==="); logs("spark-stream-processor", since="5m", tail=100)
print("\n=== asset-api ==="); logs("asset-api", since="5m", tail=50)

## 4. Wait for results in Postgres
Spark micro-batches sync to Postgres, so this can take a minute or two.

In [ ]:
assets = pd.DataFrame()
for _ in range(30):
    assets = sql(f"SELECT * FROM gold_assets WHERE tenant_id = '{TENANT}' ORDER BY fqdn")
    if len(assets):
        break
    time.sleep(10)
print(f"{len(assets)} assets")

## 5. Query tables

In [ ]:
pd.set_option("display.max_columns", None, "display.width", 250, "display.max_colwidth", 80)
assets

In [ ]:
sql(f"SELECT * FROM gold_asset_findings WHERE tenant_id = '{TENANT}'")

In [ ]:
sql(f"SELECT * FROM gold_asset_edges WHERE tenant_id = '{TENANT}'")

In [ ]:
# Row counts for all tables, including staging
tables = sql("SELECT tablename FROM pg_tables WHERE schemaname='public' ORDER BY 1")["tablename"]
pd.Series({t: int(sql(f"SELECT count(*) AS n FROM {t}")["n"][0]) for t in tables}, name="rows")

## 6. Serving API (clusters)

In [ ]:
script = f"import urllib.request as u;print(u.urlopen('http://localhost:8080/tenants/{TENANT}/clusters',timeout=30).read().decode())"
print(json.dumps(json.loads(kubectl("exec", "deploy/asset-api", "--", "python", "-c", script)), indent=2))